# CASMI 2026 · RL77 Notebook 5 — Q-Learning / RL Reward Shaping for the ChemFormer Decoder

*Assembled from **CASMI 2026 Colab Rubric.pdf** (pages 48–59). Python cells are the reviewed `experiments/runs/RL77/nb5/cell_NN.py` files; text cells are taken from the PDF.*

> **User prompt:** Write a PyTorch Q-learning / RL reward-shaping wrapper for the ChemFormer decoder that optimizes generated SMILES toward exact precursor mass and Mass2SMILES functional group consistency.

Standard teacher-forced cross-entropy ($L_{gen}$) trains the **ChemFormer** decoder to maximize token-level likelihood, which frequently produces syntactically valid SMILES that drift by a methyl or hydroxyl group from the true precursor mass or omit functional groups evidenced by MS/MS neutral losses. Wrapping the Stage 2 decoder in a **Token-Level Dueling Soft Q-Learning and Potential-Based Reward Shaping** module (`ChemFormerQDecoderWrapper`) directly optimizes generation toward exact monoisotopic precursor mass, **Mass2SMILES** 71-D functional group counts, and **MIST-CF** molecular formula priors while anchoring to the supervised Stage 2 policy via a Kullback-Leibler (KL) divergence penalty to prevent mode collapse.

## Prerequisites

| Input | Kaggle source | Provides |
| --- | --- | --- |
| Competition data | `enveda-CASMI26-molecule-id-mass-spectra` | `train.parquet`, `test.parquet`, `sample_submission.csv` |
| RL77 prerequisites | `scottweeden/casmi26-rl77-prereqs` | `rl77_env.py`, `rl77_weights.py`, `rl77_weights.json` (Evidently snapshot `rl77-v014`), upstream modules |
| Notebook 1 outputs | `scottweeden/casmi26-rl77-nb1-glmr-outputs` | Two-stage GLMR checkpoint, retrieval library, validation split, submission |
| Notebook 2 outputs | `scottweeden/casmi26-rl77-nb2-store-outputs` | Mass-sorted candidate catalog, FP16 ChemFormer memmap, FAISS index, formula index, offline wheels |
| Notebook 3 outputs | `scottweeden/casmi26-rl77-nb3-ensemble-outputs` | Siamese ECFP4 and Mass2SMILES heads, fitted blend weights, ensemble submission |
| Notebook 4 outputs | `scottweeden/casmi26-rl77-nb4-formula-outputs` | Formula priors for validation/test, fitted formula score weights, formula-constrained submission |

- **Order:** Run notebooks 1, 2, 3, 4 first and publish their outputs (`tools/kaggle_pipeline.py publish-output`).
- **Accelerator:** GPU (T4); internet on for the test run so `ensure_packages()` can reach PyPI. Offline runs install from the wheels in notebook 2's output dataset.
- **Packages:** `rdkit==2026.03.3` (the metric's canonicalizer) and `faiss-cpu` are installed if the image lacks them; `torch`, `pyarrow`, `pandas`, `scipy` come from the Kaggle image.
- **Weights:** all blend weights, reward coefficients, and tolerances come from Evidently dataset `casmi-rl77-weights` (version `rl77-v014`, id `01a1ee49-89c9-7ce4-b327-df8b7cf1d48f`) through `W("...")`.
- **Outputs:** everything written to `/kaggle/working` is published as `scottweeden/casmi26-rl77-nb5-qrl-outputs` for the next notebook.

In [ ]:
# Prerequisites: locate the RL77 helper modules, install pinned packages, load the Evidently weights snapshot
import glob, os, sys
from pathlib import Path

_lib_hits = sorted(glob.glob("/kaggle/input/**/rl77_env.py", recursive=True))
if _lib_hits:
    RL77_LIB = Path(_lib_hits[0]).parent
elif os.environ.get("RL77_LIB_DIR"):
    RL77_LIB = Path(os.environ["RL77_LIB_DIR"])  # e.g. experiments/runs/RL77/build/datasets/casmi26-rl77-prereqs
else:
    raise FileNotFoundError("Attach the scottweeden/casmi26-rl77-prereqs dataset or set RL77_LIB_DIR.")
sys.path.insert(0, str(RL77_LIB))

import rl77_env as ENV
ENV.ensure_packages()                      # rdkit==2026.03.3 + faiss-cpu (offline wheels when attached)
from rl77_weights import load_rl77_weights
W = load_rl77_weights()                    # every blend weight / reward coefficient in this notebook reads W("...")
from rl77_nb1_glmr import *  # nb1: CASMI26 RL77 NB1 GLMR
from rl77_nb2_store import *  # nb2: CASMI26 RL77 NB2 Offline Store
from rl77_nb3_ensemble import *  # nb3: CASMI26 RL77 NB3 Ensemble
from rl77_nb4_formula import *  # nb4: CASMI26 RL77 NB4 Formula Filter
ENV.log(f"Competition data: {ENV.competition_dir()} | outputs: {ENV.work_dir()}")


## Cell 1: Dense Prefix Potential & Terminal Chemical Reward Engine

To overcome the sparse-reward problem (where RDKit can only parse a SMILES string after the `[EOS]` token is emitted), `CASMI26ChemicalRewardEngine` pairs a **dense heavy-atom mass potential** $\Phi(s_t)$ at every token step $t$ with a **multi-objective terminal reward** $R_{\text{term}}(y)$ at `[EOS]`. By Ng et al.'s potential-based reward shaping theorem, intermediate shaping rewards $F(s_t, s_{t+1}) = \gamma\Phi(s_{t+1}) - \Phi(s_t)$ accelerate Q-value convergence without altering the optimal chemical policy.

> **Review fix:** the notebook-1 tokenizer is character-level, so the `"Cl"` / `"Br"` keys never matched: chlorine counted as 12 Da (its `C`) and bromine as 0. Per-character increments (`l`, `B`, `r`) keep the heavy-atom lower bound exact, and explicit `H` is counted. All reward and potential coefficients come from the Evidently weights (`nb5.reward.*`, `nb5.potential.*`).

In [ ]:
# [Colab Q-RL Cell 1] Dense Prefix Mass Potential & Multi-Objective Terminal Chemical Reward Engine
import math
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# Approximate minimum mass contribution (atom + minimum typical implicit H) per SMILES character
TOKEN_HEAVY_MASS_LOWER_BOUND = {
    "C": 12.0000, "c": 12.0000,
    "N": 14.0031, "n": 14.0031,
    "O": 15.9949, "o": 15.9949,
    "S": 31.9721, "s": 31.9721,
    "P": 30.9738, "p": 30.9738,
    "F": 18.9984,
    "Cl": 34.9689,
    "Br": 78.9183,
    "I": 126.9045,
    # The notebook-1 tokenizer is character-level: "Cl" arrives as "C" + "l" and "Br" as "B" + "r".
    # The two-letter keys above never matched, so Cl counted as 12 Da and Br as 0. Per-character
    # increments keep the lower bound exact: C + l = Cl, B (boron) + r = Br.
    "l": 34.9689 - 12.0000,
    "B": 11.0093,
    "r": 78.9183 - 11.0093,
    "H": 1.0078,  # explicit bracket hydrogens, e.g. [nH], [NH+]
}


class CASMI26ChemicalRewardEngine:
    """
    Computes:
      1. Dense prefix potential Phi(s_t) from partial heavy-atom mass accumulation and bracket/ring balance.
      2. Terminal reward R_term(y) at [EOS] combining:
         - r_valid:   RDKit 2026.03.3 parseability + tautomer canonicalization
         - r_mass:    Gaussian + Laplace reward on exact monoisotopic mass vs. neutral precursor mass
         - r_fg71:    Cosine + L1 agreement with Mass2SMILES 71-D functional group & heteroatom predictions
         - r_formula: Bonus if Hill formula matches one of the top-3 MIST-CF formula priors
         - r_emb:     Unimodal cosine similarity cos(E^m_gen, E^m_target) in ChemFormer space
    """

    def __init__(
        self,
        tokenizer,
        canonicalize_fn,
        fg_extractor_fn,
        mass_sigma_da: float = W("nb5.reward.mass_sigma_da"),
        coarse_mass_scale_da: float = W("nb5.reward.coarse_mass_scale_da"),
    ):
        self.tokenizer = tokenizer
        self.canonicalize_fn = canonicalize_fn
        self.fg_extractor_fn = fg_extractor_fn
        self.mass_sigma_da = mass_sigma_da
        self.coarse_mass_scale_da = coarse_mass_scale_da

        # Precompute per-token heavy-atom mass table aligned with tokenizer vocabulary
        self.token_masses = np.zeros(len(tokenizer), dtype=np.float32)
        for tok_str, idx in tokenizer.stoi.items():
            if tok_str in TOKEN_HEAVY_MASS_LOWER_BOUND:
                self.token_masses[idx] = TOKEN_HEAVY_MASS_LOWER_BOUND[tok_str]

    def compute_prefix_mass_and_potential(
        self,
        token_ids: torch.Tensor,       # (B, L)
        target_neutral_mass: torch.Tensor,  # (B,)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Computes cumulative heavy-atom mass at every token step t in 1..L and returns:
          - rem_mass_ratio: (B, L) normalized remaining mass budget (m_target - m_heavy_t) / m_target
          - phi_potential:  (B, L) potential function Phi(s_t) penalizing mass budget overruns
        """
        device = token_ids.device
        tok_mass_t = torch.tensor(self.token_masses, device=device)[token_ids]  # (B, L)
        cum_heavy_mass = torch.cumsum(tok_mass_t, dim=1)                        # (B, L)

        target_m = target_neutral_mass.unsqueeze(1).clamp(min=50.0)             # (B, 1)
        # Since hydrogens are implicit in SMILES, heavy atoms typically account for ~90-94% of total mass
        expected_heavy_target = W("nb5.potential.heavy_fraction") * target_m
        rem_mass = target_m - cum_heavy_mass
        rem_mass_ratio = torch.clamp(rem_mass / target_m, min=-1.0, max=1.0)

        # Smooth potential peaks as cumulative heavy mass approaches expected_heavy_target,
        # and drops sharply if heavy-atom mass alone exceeds target_neutral_mass + 1.0 Da
        overrun_da = F.relu(cum_heavy_mass - (target_m + W("nb5.potential.overrun_margin_da")))
        underrun_frac = torch.abs(cum_heavy_mass - expected_heavy_target) / target_m
        phi_potential = torch.exp(-W("nb5.potential.underrun_decay") * underrun_frac) - W("nb5.potential.overrun_penalty") * overrun_da

        return rem_mass_ratio, phi_potential

    @torch.no_grad()
    def evaluate_terminal_smiles_batch(
        self,
        generated_seqs: List[List[int]],
        target_neutral_masses: np.ndarray,          # (B,)
        target_fg71_preds: np.ndarray,              # (B, 71)
        formula_priors_batch: Optional[List[List[str]]] = None,
        mol_encoder: Optional[nn.Module] = None,
        target_mol_embs: Optional[torch.Tensor] = None,  # (B, 256) ground-truth or Stage-1 anchor
        device: torch.device = torch.device("cpu"),
    ) -> Tuple[torch.Tensor, List[Optional[str]], List[Optional[str]]]:
        """
        Computes scalar terminal reward R_term in [-1.0, +5.0] for each decoded sequence in batch.
        """
        B = len(generated_seqs)
        rewards = np.zeros(B, dtype=np.float32)
        canon_smiles_out: List[Optional[str]] = []
        ik14_out: List[Optional[str]] = []

        valid_indices = []
        valid_tok_Batch = []

        for b in range(B):
            raw_smi = self.tokenizer.decode(generated_seqs[b])
            canon_smi, ik14 = self.canonicalize_fn(raw_smi)
            canon_smiles_out.append(canon_smi)
            ik14_out.append(ik14)

            if canon_smi is None:
                rewards[b] = W("nb5.reward.invalid")  # Invalid SMILES syntax or rejected valence
                continue

            mol = Chem.MolFromSmiles(canon_smi)
            if mol is None:
                rewards[b] = W("nb5.reward.invalid")
                continue

            # 1. Chemical Validity Base Reward (+0.50)
            r_total = W("nb5.reward.valid_base")

            # 2. Exact Monoisotopic Precursor Mass Reward (up to +2.00)
            # Combines a coarse exponential funnel (guides within +/- 14 Da) and a fine sub-15 ppm peak
            gen_mass = float(Descriptors.ExactMolWt(mol))
            target_m = float(target_neutral_masses[b])
            mass_diff_da = abs(gen_mass - target_m)
            ppm_err = (mass_diff_da / max(target_m, 100.0)) * 1e6

            r_mass_coarse = math.exp(-mass_diff_da / self.coarse_mass_scale_da)
            r_mass_fine = math.exp(-0.5 * (mass_diff_da / self.mass_sigma_da) ** 2)
            r_mass = W("nb5.reward.mass_coarse") * r_mass_coarse + W("nb5.reward.mass_fine") * r_mass_fine
            r_total += r_mass

            # 3. MIST-CF / SIRIUS Molecular Formula Match Bonus (+0.75)
            if formula_priors_batch is not None and b < len(formula_priors_batch):
                gen_formula = rdMolDescriptors.CalcMolFormula(mol)
                priors = formula_priors_batch[b]
                if priors and gen_formula == priors[0]:
                    r_total += W("nb5.reward.formula_top1")
                elif priors and gen_formula in priors[:3]:
                    r_total += W("nb5.reward.formula_top3")

            # 4. Mass2SMILES 71-D Functional Group + Heteroatom Consistency Reward (up to +1.00)
            _, gen_fg71 = self.fg_extractor_fn(canon_smi)
            if gen_fg71 is not None:
                pred_fg71 = target_fg71_preds[b]
                # Separate functional groups (0..59) and heteroatoms/rings (60..70)
                fg_l1 = float(np.mean(np.abs(gen_fg71[:60] - pred_fg71[:60])))
                hetero_l1 = float(np.mean(np.abs(gen_fg71[60:] - pred_fg71[60:])))
                r_fg = (W("nb5.reward.fg_weight") * math.exp(-W("nb5.reward.fg_decay") * fg_l1)
                        + W("nb5.reward.hetero_weight") * math.exp(-W("nb5.reward.hetero_decay") * hetero_l1))
                r_total += r_fg

            rewards[b] = r_total
            if mol_encoder is not None and target_mol_embs is not None:
                valid_indices.append(b)
                valid_tok_Batch.append(self.tokenizer.encode(canon_smi, add_cls=True))

        # 5. Unimodal ChemFormer Embedding Cosine Similarity Bonus (up to +0.75)
        if valid_indices and mol_encoder is not None and target_mol_embs is not None:
            tok_tensor = torch.stack(valid_tok_Batch).to(device)
            _, e_m_gen = mol_encoder(tok_tensor)  # (N_val, 256)
            target_sub = target_mol_embs[valid_indices].to(device)
            cos_sims = torch.sum(e_m_gen * target_sub, dim=-1).clamp(min=0.0).cpu().numpy()
            for idx_pos, b_idx in enumerate(valid_indices):
                rewards[b_idx] += W("nb5.reward.embedding_bonus") * float(cos_sims[idx_pos])

        return torch.tensor(rewards, dtype=torch.float32, device=device), canon_smiles_out, ik14_out

## Cell 2: Dueling Soft Q-Learning Wrapper (`ChemFormerQDecoderWrapper`)

To make the decoder explicitly mass-aware at every generation step, `ChemFormerQDecoderWrapper` extracts the causal Transformer hidden state $\mathbf{h}_t \in \mathbb{R}^{256}$, concatenates an encoding of the **remaining precursor mass budget** $(m_{\text{target}} - m_{\text{heavy}}(s_t))$, and passes the fused state through a **Dueling Q-Network** $\left(Q(s_t, a_t) = V(s_t) + A(s_t, a_t) - \overline{A}(s_t)\right)$.

In [ ]:
# [Colab Q-RL Cell 2] Mass-Conditioned Dueling Q-Network Wrapper for ChemFormerMolecularDecoder
class ChemFormerQDecoderWrapper(nn.Module):
    """
    Wraps ChemFormerMolecularDecoder with:
      1. Remaining-mass sinusoidal state conditioning (informs policy & Q-head how many Daltons remain).
      2. Dueling Action-Value Head Q(s_t, a_t) = V(s_t) + (A(s_t, a_t) - mean_a' A(s_t, a')).
      3. EMA Target Q-Network parameters for stable Bellman temporal-difference backups.
    """

    def __init__(
        self,
        base_decoder: nn.Module,
        d_model: int = 256,
        mass_emb_dim: int = 32,
    ):
        super().__init__()
        self.base_decoder = base_decoder
        self.vocab_size = base_decoder.out_proj.out_features
        self.d_model = d_model

        # Project [remaining_mass_ratio, cumulative_mass / 1000.0] into mass_emb_dim
        self.mass_budget_proj = nn.Sequential(
            nn.Linear(2, mass_emb_dim),
            nn.LayerNorm(mass_emb_dim),
            nn.GELU(),
        )

        # Dueling Q-Network: State Value V(s_t) and Token Advantage A(s_t, a_t)
        fused_dim = d_model + mass_emb_dim
        self.value_head = nn.Sequential(
            nn.Linear(fused_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )
        self.advantage_head = nn.Sequential(
            nn.Linear(fused_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, self.vocab_size),
        )

        # Target network for stable TD backups (updated via Polyak averaging)
        self.target_value_head = nn.Sequential(
            nn.Linear(fused_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )
        self.target_advantage_head = nn.Sequential(
            nn.Linear(fused_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, self.vocab_size),
        )
        self.sync_target_q_heads(tau=1.0)

    @torch.no_grad()
    def sync_target_q_heads(self, tau: float = 0.005) -> None:
        """Polyak soft update of target Q-heads: theta_targ <- tau * theta + (1 - tau) * theta_targ."""
        for p, p_targ in zip(self.value_head.parameters(), self.target_value_head.parameters()):
            p_targ.data.mul_(1.0 - tau).add_(p.data, alpha=tau)
        for p, p_targ in zip(self.advantage_head.parameters(), self.target_advantage_head.parameters()):
            p_targ.data.mul_(1.0 - tau).add_(p.data, alpha=tau)

    def extract_decoder_hidden_states(
        self, tgt_ids: torch.Tensor, memory: torch.Tensor
    ) -> torch.Tensor:
        """Runs the causal Transformer stack of base_decoder and returns hidden states (B, Q, D)."""
        B, Q = tgt_ids.shape
        positions = torch.arange(Q, device=tgt_ids.device).unsqueeze(0).expand(B, Q)
        tgt = self.base_decoder.token_emb(tgt_ids) + self.base_decoder.pos_emb(positions)
        causal_mask = torch.triu(torch.ones(Q, Q, dtype=torch.bool, device=tgt_ids.device), diagonal=1)
        tgt_pad_mask = tgt_ids == self.base_decoder.pad_id
        h_dec = self.base_decoder.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_pad_mask,
        )
        return h_dec

    def forward_policy_and_q(
        self,
        tgt_ids: torch.Tensor,            # (B, Q)
        memory: torch.Tensor,             # (B, P, D) cross-fused H^ca
        rem_mass_ratio: torch.Tensor,     # (B, Q)
        target_neutral_mass: torch.Tensor,  # (B,)
        use_target_q: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns:
          - policy_logits: (B, Q, V) from base_decoder.out_proj
          - q_values:      (B, Q, V) action-values Q(s_t, a_t) for every token in vocabulary
          - state_values:  (B, Q, 1) V(s_t)
        """
        h_dec = self.extract_decoder_hidden_states(tgt_ids, memory)
        policy_logits = self.base_decoder.out_proj(h_dec)

        norm_target_m = (target_neutral_mass / 1000.0).unsqueeze(1).expand_as(rem_mass_ratio)
        mass_feats = torch.stack([rem_mass_ratio, norm_target_m], dim=-1)  # (B, Q, 2)
        mass_emb = self.mass_budget_proj(mass_feats)                        # (B, Q, mass_emb_dim)

        fused_state = torch.cat([h_dec, mass_emb], dim=-1)                  # (B, Q, D + mass_emb_dim)

        if use_target_q:
            v_s = self.target_value_head(fused_state)
            adv = self.target_advantage_head(fused_state)
        else:
            v_s = self.value_head(fused_state)
            adv = self.advantage_head(fused_state)

        q_values = v_s + (adv - adv.mean(dim=-1, keepdim=True))
        return policy_logits, q_values, v_s

## Cell 3: Stage 2.5 Hybrid Soft Q-Learning + KL-Anchored Actor-Critic Trainer

Training an autoregressive chemical language model purely with policy gradients or unconstrained Q-learning quickly collapses syntactic SMILES validity. This trainer combines three complementary objectives:

1. **Shaped Soft Bellman TD Loss ($\mathcal{L}_Q$):** Trains $Q_\theta(s_t, a_t)$ on both teacher-forced ground-truth trajectories and on-policy sampled trajectories using dense potential shaping $r_t = \gamma\Phi(s_{t+1}) - \Phi(s_t)$ for $t < T$ and $r_T = R_{\text{term}}(y) - \Phi(s_T)$ at `[EOS]`.
2. **Conservative Q-Regularization ($\mathcal{L}_{\text{CQL}}$):** Penalizes out-of-distribution chemical token actions so unvisited invalid SMILES transitions do not receive overestimated Q-values.
3. **KL-Anchored Advantage Actor Loss ($\mathcal{L}_\pi$):** Updates the `ChemFormerMolecularDecoder` weights toward high-advantage mass- and functional-group-consistent tokens while penalizing divergence $\mathrm{KL}(\pi_\phi \,\|\, \pi_{\text{ref}})$ from the frozen Stage 2 checkpoint.

> **Review fix:** the bootstrap target $Q_{\text{targ}}(s_{t+1}, \cdot)$ was computed by feeding `act_ids` (the trajectory without `[BOS]`) as a sequence, which shifted every position embedding and dropped the BOS context; it is now the full-trajectory target-network output at position $t+1$. Neutral mass uses `neutral_mass_from_precursor()`, and γ, α, CQL, KL, anchor, Polyak, rollout temperature, and the expert reward come from the Evidently weights.

> **Review fix (after the nb5 Kaggle run):** the actor's advantage was the critic's $Q - V$. While the dueling heads are immature, that signal is uncorrelated with the actual outcome. In the first full run the policy drifted and rollouts collapsed to invalid SMILES: the mean terminal reward went from −0.28 (best −0.11 around step 250) to −0.96 by step 4,885, and Q-guided beams were 49% valid with 0% inside the mass window. The actor now uses REINFORCE with a batch baseline, $\hat A = (R(y) - \bar R)/\sigma_R$ shared by every token of $y$, so an all-invalid batch gives no push. The Q-heads still learn by the shaped soft Bellman TD loss and still guide beam search.

In [ ]:
# [Colab Q-RL Cell 3] Stage 2.5 Soft Q-Learning + Conservative Q + KL-Anchored Policy Optimization
import copy


class GLMRStage25QLearningTrainer:
    """
    Fine-tunes the Stage 2 Spec-Mol Cross-Fusion Decoder using Dueling Soft Q-Learning
    and KL-regularized Advantage Actor-Critic updates.
    """

    def __init__(
        self,
        q_decoder: ChemFormerQDecoderWrapper,
        reward_engine: CASMI26ChemicalRewardEngine,
        gamma: float = W("nb5.rl.gamma"),
        alpha_ent: float = W("nb5.rl.alpha_ent"),
        cql_weight: float = W("nb5.rl.cql_weight"),
        kl_beta: float = W("nb5.rl.kl_beta"),
        sup_anchor_weight: float = W("nb5.rl.sup_anchor_weight"),
        polyak_tau: float = W("nb5.rl.polyak_tau"),
    ):
        self.q_decoder = q_decoder
        self.reward_engine = reward_engine
        self.gamma = gamma
        self.alpha_ent = alpha_ent
        self.cql_weight = cql_weight
        self.kl_beta = kl_beta
        self.sup_anchor_weight = sup_anchor_weight
        self.polyak_tau = polyak_tau

        # Freeze a copy of the Stage 2 supervised decoder as the KL reference policy pi_ref
        self.ref_decoder = copy.deepcopy(q_decoder.base_decoder).eval()
        for p in self.ref_decoder.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def sample_on_policy_rollouts(
        self,
        memory: torch.Tensor,              # (B, P, D) H^ca
        target_neutral_mass: torch.Tensor, # (B,)
        max_len: int = 96,
        temperature: float = W("nb5.rl.rollout_temperature"),
    ) -> Tuple[torch.Tensor, List[List[int]]]:
        """Samples on-policy SMILES trajectories y ~ pi_phi( . | H^ca) for RL exploration."""
        self.q_decoder.eval()
        B = memory.size(0)
        device = memory.device
        bos_id = self.q_decoder.base_decoder.bos_id
        eos_id = self.q_decoder.base_decoder.eos_id
        pad_id = self.q_decoder.base_decoder.pad_id

        seqs = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_len - 1):
            logits = self.q_decoder.base_decoder(seqs, memory)[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)
            next_tok = torch.where(finished, torch.tensor(pad_id, device=device), next_tok)
            seqs = torch.cat([seqs, next_tok.unsqueeze(1)], dim=1)
            finished = finished | (next_tok == eos_id)
            if torch.all(finished):
                break

        decoded_lists = [row.cpu().tolist() for row in seqs]
        self.q_decoder.train()
        return seqs, decoded_lists

    def compute_shaped_td_and_actor_losses(
        self,
        traj_ids: torch.Tensor,             # (B, Q) token trajectory [BOS, y_1, ..., EOS, PAD]
        memory: torch.Tensor,               # (B, P, D) H^ca
        target_neutral_mass: torch.Tensor,  # (B,)
        terminal_rewards: torch.Tensor,     # (B,) R_term(y)
        sequence_advantages: Optional[torch.Tensor] = None,  # (B,) normalized Monte Carlo advantages for the actor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Computes:
          1. Soft Bellman TD loss + CQL penalty on Q(s_t, a_t)
          2. Advantage-weighted Actor loss + KL(pi_phi || pi_ref) penalty
        """
        pad_id = self.q_decoder.base_decoder.pad_id
        eos_id = self.q_decoder.base_decoder.eos_id

        inp_ids = traj_ids[:, :-1]   # (B, T) states s_t
        act_ids = traj_ids[:, 1:]    # (B, T) actions a_t
        valid_mask = (act_ids != pad_id).float()  # (B, T)

        # Compute dense prefix potentials Phi(s_t) and Phi(s_{t+1})
        rem_ratio_all, phi_all = self.reward_engine.compute_prefix_mass_and_potential(
            traj_ids, target_neutral_mass
        )
        rem_s = rem_ratio_all[:, :-1]
        rem_next = rem_ratio_all[:, 1:]
        phi_s = phi_all[:, :-1]
        phi_next = phi_all[:, 1:]

        # Forward current policy & Q-network on s_t
        policy_logits, q_all, v_s = self.q_decoder.forward_policy_and_q(
            inp_ids, memory, rem_s, target_neutral_mass, use_target_q=False
        )
        q_taken = q_all.gather(dim=-1, index=act_ids.unsqueeze(-1)).squeeze(-1)  # (B, T)

        # Forward target Q-network on s_{t+1} for Soft Bellman backup
        with torch.no_grad():
            # Q_targ(s_{t+1}, .) is the full-trajectory output at position t+1. The PDF fed act_ids
            # (the trajectory without BOS) as if it were a sequence, which shifted every position
            # embedding and dropped the BOS context from the bootstrap target.
            _, q_full_targ, _ = self.q_decoder.forward_policy_and_q(
                traj_ids, memory, rem_ratio_all, target_neutral_mass, use_target_q=True
            )
            q_next_all = q_full_targ[:, 1:, :]
            # Soft value V_targ(s_{t+1}) = alpha * logsumexp(Q_targ(s_{t+1}, a') / alpha)
            v_next_soft = self.alpha_ent * torch.logsumexp(
                q_next_all / self.alpha_ent, dim=-1
            )

            # Identify terminal step for each sequence (where act_ids == eos_id or last valid step)
            is_terminal = (act_ids == eos_id) | (
                torch.cat([valid_mask[:, 1:], torch.zeros_like(valid_mask[:, :1])], dim=1) == 0.0
            )
            is_term_f = is_terminal.float()

            # Potential-based reward shaping: r_t = gamma * Phi(s_{t+1}) - Phi(s_t) on non-terminal steps,
            # and r_T = R_term(y) - Phi(s_T) on the terminal step
            step_rewards = (1.0 - is_term_f) * (self.gamma * phi_next - phi_s) + is_term_f * (
                terminal_rewards.unsqueeze(1) - phi_s
            )
            td_targets = step_rewards + (1.0 - is_term_f) * self.gamma * v_next_soft

            # Reference policy logits for KL regularization
            ref_logits = self.ref_decoder(inp_ids, memory)

        # 1. Huber TD Loss + Conservative Q-Learning (CQL) penalty
        td_loss = F.smooth_l1_loss(q_taken, td_targets, reduction="none")
        cql_penalty = torch.logsumexp(q_all, dim=-1) - q_taken
        q_loss = torch.sum((td_loss + self.cql_weight * cql_penalty) * valid_mask) / valid_mask.sum().clamp(min=1.0)

        # 2. Advantage-Weighted Policy Gradient + KL(pi_phi || pi_ref)
        log_probs = F.log_softmax(policy_logits, dim=-1)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        probs = F.softmax(policy_logits, dim=-1)

        taken_log_prob = log_probs.gather(dim=-1, index=act_ids.unsqueeze(-1)).squeeze(-1)
        if sequence_advantages is not None:
            # REINFORCE with a batch baseline: every token of y shares (R(y) - mean R) / std R. Kaggle nb5 v1 used the
            # critic's Q - V here; with immature dueling heads that advantage was uncorrelated with the real outcome,
            # the actor drifted, and rollouts collapsed to invalid SMILES (mean reward -0.28 -> -0.96 over 4,885 steps).
            norm_advantages = sequence_advantages.detach().unsqueeze(1).expand_as(valid_mask)
        else:
            advantages = (q_taken - v_s.squeeze(-1)).detach()
            # Normalize advantages across valid tokens in batch
            adv_mean = (advantages * valid_mask).sum() / valid_mask.sum().clamp(min=1.0)
            adv_std = torch.sqrt(
                ((advantages - adv_mean).pow(2) * valid_mask).sum() / valid_mask.sum().clamp(min=1.0) + 1e-6
            )
            norm_advantages = torch.clamp((advantages - adv_mean) / adv_std, min=-3.0, max=3.0)

        pg_loss = -taken_log_prob * norm_advantages
        kl_div = torch.sum(probs * (log_probs - ref_log_probs), dim=-1)
        actor_loss = torch.sum((pg_loss + self.kl_beta * kl_div) * valid_mask) / valid_mask.sum().clamp(min=1.0)

        return q_loss, actor_loss

    def train_rl_step(
        self,
        batch: Dict[str, torch.Tensor],
        glmr_model: nn.Module,
        fg_head: nn.Module,
        subformula_filter,
        generate_formula_priors_fn,
        retrieve_topk_fn,
        lib_masses: np.ndarray,
        lib_embs: torch.Tensor,
        adduct_shifts: Dict[str, float],
        optimizer: torch.optim.Optimizer,
        device: torch.device,
    ) -> Dict[str, float]:
        """Executes one complete hybrid Off-Policy (Expert) + On-Policy (Rollout) Q-RL optimization step."""
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)

        with torch.no_grad():
            h_s_mol, e_s_mol, _ = glmr_model.encode_molecule_spectra(batch)
            fg_71_pred, _ = fg_head(h_s_mol)
            _, target_mol_embs = glmr_model.mol_encoder(batch["mol_tokens_cls"])

            h_m_k, _ = retrieve_topk_fn(
                e_s_mol, batch["precursor_mz"], batch["primary_adduct"], lib_masses, lib_embs
            )
            h_ca = glmr_model.cross_fusion(h_s_mol, h_m_k)

            # Infer target neutral mass per molecule in batch
            B = h_ca.size(0)
            neutral_masses = []
            for b in range(B):
                pmz = float(batch["precursor_mz"][b].item())
                # adduct_shifts kept for the PDF signature; the helper handles formate/acetate and [2M+X]
                neutral_masses.append(neutral_mass_from_precursor(
                    pmz, str(batch["primary_adduct"][b]), str(batch["ionization_mode"][b])
                ))
            target_neutral_mass_t = torch.tensor(neutral_masses, dtype=torch.float32, device=device)

            formula_priors = generate_formula_priors_fn(
                batch, subformula_filter, fg_71_pred_batch=fg_71_pred
            )

        # A. On-policy rollout trajectories y_hat ~ pi_phi(. | H^ca)
        rollout_seqs, rollout_lists = self.sample_on_policy_rollouts(
            h_ca, target_neutral_mass_t, max_len=96
        )
        rollout_rewards, _, _ = self.reward_engine.evaluate_terminal_smiles_batch(
            generated_seqs=rollout_lists,
            target_neutral_masses=np.asarray(neutral_masses, dtype=np.float32),
            target_fg71_preds=fg_71_pred.cpu().numpy(),
            formula_priors_batch=formula_priors,
            mol_encoder=glmr_model.mol_encoder,
            target_mol_embs=target_mol_embs,
            device=device,
        )

        # Monte Carlo sequence advantages for the actor; an all-invalid (or all-equal) batch gives no push at all.
        r_std = rollout_rewards.std(unbiased=False)
        mc_adv = torch.zeros_like(rollout_rewards) if r_std < 1e-6 else torch.clamp(
            (rollout_rewards - rollout_rewards.mean()) / (r_std + 1e-6), -3.0, 3.0
        )
        q_loss_on, actor_loss_on = self.compute_shaped_td_and_actor_losses(
            rollout_seqs, h_ca, target_neutral_mass_t, rollout_rewards, sequence_advantages=mc_adv
        )

        # B. Expert ground-truth trajectory anchor (replay buffer / imitation anchor)
        expert_seqs = batch["mol_tokens_dec"]
        expert_rewards = torch.full((B,), W("nb5.rl.expert_reward"), dtype=torch.float32, device=device)
        q_loss_exp, _ = self.compute_shaped_td_and_actor_losses(
            expert_seqs, h_ca, target_neutral_mass_t, expert_rewards
        )

        # Supervised token anchor loss on ground-truth SMILES to preserve syntactic grammar
        sup_logits = self.q_decoder.base_decoder(expert_seqs[:, :-1], h_ca)
        sup_loss = F.cross_entropy(
            sup_logits.reshape(-1, sup_logits.size(-1)),
            expert_seqs[:, 1:].reshape(-1),
            ignore_index=self.q_decoder.base_decoder.pad_id,
        )

        total_loss = (
            0.5 * (q_loss_on + q_loss_exp)
            + actor_loss_on
            + self.sup_anchor_weight * sup_loss
        )

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_decoder.parameters(), 1.0)
        optimizer.step()

        self.q_decoder.sync_target_q_heads(tau=self.polyak_tau)

        return {
            "total_loss": float(total_loss.item()),
            "q_loss": float(0.5 * (q_loss_on + q_loss_exp).item()),
            "actor_loss": float(actor_loss_on.item()),
            "mean_rollout_reward": float(rollout_rewards.mean().item()),
        }

## Cell 4: Mass-Constrained Q-Guided Beam Search

During inference on `test.parquet`, `q_guided_beam_search_decode` replaces standard log-probability beam search. At every step $t$, candidate token extensions are scored by the joint **Actor-Critic objective**:

$$\text{Score}(s_t, a_t) = \log \pi_\phi(a_t \mid s_t, \mathbf{H}^{ca}) + \beta_Q\, Q_\theta(s_t, a_t, m_{\text{rem}})$$

Any partial beam whose accumulated heavy-atom mass exceeds `target_neutral_mass + 2.0 Da` is immediately pruned, and all completed beams are re-ranked by `CASMI26ChemicalRewardEngine` so the top valid de novo isomers can both anchor GLMR Stage 2 unimodal similarity and populate novel scaffold slots (Tier 3).

> **Review fix:** beam width, β, pruning margin, and the reward mix come from the Evidently weights, and per-beam rewards are returned for the centroid below. The chrysin/paracetamol smoke test is replaced by the real run in Cell 5.

In [ ]:
# [Colab Q-RL Cell 4] Mass-Constrained Q-Guided Beam Search & End-to-End Smoke Test
@torch.no_grad()
def q_guided_beam_search_decode(
    q_decoder: ChemFormerQDecoderWrapper,
    reward_engine: CASMI26ChemicalRewardEngine,
    memory: torch.Tensor,                 # (B, P, D) H^ca
    target_neutral_masses: np.ndarray,     # (B,)
    fg_71_preds: np.ndarray,               # (B, 71)
    formula_priors_batch: Optional[List[List[str]]] = None,
    beam_size: int = W.int("nb5.beam.size"),
    q_weight_beta: float = W("nb5.beam.q_weight_beta"),
    max_len: int = W.int("nb1.decode.max_len"),
    reward_lists: Optional[List[List[float]]] = None,
) -> Tuple[List[List[str]], List[List[str]]]:
    """
    Executes Mass-Constrained Q-Guided Beam Search per molecule_id and returns:
      - top_valid_smiles_per_mol: List[List[str]] of canonicalized, mass-consistent de novo SMILES
      - top_valid_ik14_per_mol:   List[List[str]] of corresponding unique InChIKey14 blocks
    """
    q_decoder.eval()
    device = memory.device
    bos_id = q_decoder.base_decoder.bos_id
    eos_id = q_decoder.base_decoder.eos_id
    token_masses = reward_engine.token_masses

    B = memory.size(0)
    all_mol_smiles: List[List[str]] = []
    all_mol_ik14s: List[List[str]] = []

    for b in range(B):
        mem_b = memory[b : b + 1]  # (1, P, D)
        target_m = float(target_neutral_masses[b])
        target_m_t = torch.tensor([target_m], dtype=torch.float32, device=device)

        # Each beam stores: (token_list, cumulative_score, cumulative_heavy_mass)
        beams: List[Tuple[List[int], float, float]] = [([bos_id], 0.0, 0.0)]
        completed_seqs: List[Tuple[List[int], float]] = []

        for _ in range(max_len - 1):
            active_seqs = [seq for seq, _, _ in beams if seq[-1] != eos_id]
            if not active_seqs:
                break

            # Batched evaluation of all active beams for molecule b
            n_act = len(active_seqs)
            inp_t = torch.tensor(active_seqs, dtype=torch.long, device=device)
            mem_exp = mem_b.expand(n_act, -1, -1)
            tm_exp = target_m_t.expand(n_act)

            rem_ratio, _ = reward_engine.compute_prefix_mass_and_potential(inp_t, tm_exp)
            policy_logits, q_vals, _ = q_decoder.forward_policy_and_q(
                inp_t, mem_exp, rem_ratio, tm_exp, use_target_q=False
            )

            last_log_probs = F.log_softmax(policy_logits[:, -1, :], dim=-1)  # (n_act, V)
            last_q_vals = q_vals[:, -1, :]                                   # (n_act, V)
            joint_step_scores = last_log_probs + q_weight_beta * torch.tanh(last_q_vals)

            next_candidates: List[Tuple[List[int], float, float]] = []
            act_ptr = 0

            for seq, cum_score, cum_mass in beams:
                if seq[-1] == eos_id:
                    completed_seqs.append((seq, cum_score / len(seq)))
                    continue

                row_scores = joint_step_scores[act_ptr]
                act_ptr += 1

                topk_sc, topk_tok = torch.topk(row_scores, k=beam_size * 2)
                for sc, tok_id in zip(topk_sc.tolist(), topk_tok.tolist()):
                    new_mass = cum_mass + float(token_masses[tok_id])
                    # Hard physical pruning: heavy atoms alone cannot exceed neutral_mass + 2.0 Da
                    if new_mass > (target_m + W("nb5.beam.mass_prune_margin_da")):
                        continue
                    next_candidates.append((seq + [tok_id], cum_score + sc, new_mass))

            if not next_candidates:
                break
            next_candidates.sort(key=lambda x: x[1] / len(x[0]), reverse=True)
            beams = next_candidates[:beam_size]
            if len(completed_seqs) >= beam_size * 2:
                break

        for seq, cum_score, _ in beams:
            completed_seqs.append((seq, cum_score / len(seq)))

        # Re-rank all completed beams using the exact chemical reward engine (Mass + 71-D FG + Formula)
        seq_list = [s for s, _ in completed_seqs]
        beam_rewards, canon_smis, ik14s = reward_engine.evaluate_terminal_smiles_batch(
            generated_seqs=seq_list,
            target_neutral_masses=np.full(len(seq_list), target_m, dtype=np.float32),
            target_fg71_preds=np.repeat(fg_71_preds[b : b + 1], len(seq_list), axis=0),
            formula_priors_batch=[formula_priors_batch[b]] * len(seq_list) if formula_priors_batch else None,
            device=device,
        )

        # Combine normalized beam search trajectory score with terminal chemical reward
        final_beam_Order = np.argsort(
            -(beam_rewards.cpu().numpy() + W("nb5.beam.reward_mix") * np.array([sc for _, sc in completed_seqs]))
        )

        rewards_np = beam_rewards.cpu().numpy()
        seen_ik14 = set()
        mol_valid_smis, mol_valid_iks, mol_rewards = [], [], []
        for idx in final_beam_Order:
            c_smi, ik = canon_smis[idx], ik14s[idx]
            if c_smi is not None and ik is not None and ik not in seen_ik14:
                seen_ik14.add(ik)
                mol_valid_smis.append(c_smi)
                mol_valid_iks.append(ik)
                mol_rewards.append(float(rewards_np[idx]))

        all_mol_smiles.append(mol_valid_smis)
        all_mol_ik14s.append(mol_valid_iks)
        if reward_lists is not None:
            reward_lists.append(mol_rewards)

    return all_mol_smiles, all_mol_ik14s

## Cell 5: Stage 2.5 Training on the Competition Data, Q-Guided Validation, Test Submission

The full run: fine-tune notebook 1's decoder with the Stage 2.5 trainer on notebook 1's training split within a GPU time budget, then compare on the same held-out timsTOF molecules notebook 1's pipeline (with its adopted re-rank weights) against Q-guided beams with the reward-weighted centroid below (MRR@25, validity, mass agreement, exact top-1). Candidates matching notebook 4's top formula priors rank first when notebook 4 adopted formula tiers (`nb4.priors.rank_formulas` > 0). The centroid blend α and the Q-beam insert rank are notebook 5's own weights (`nb5.rerank.*`), cross-fitted on the held-out molecules and written to `fitted_weights.json`; the test submission uses the fit only when it beats the prior on the held-out halves (the rule `tools/evidently_weights.py adopt` applies). A best-checkpoint guard keeps the decoder and Q-heads with the highest 100-step moving-average rollout reward and restores them before validation. Outputs become the `casmi26-rl77-nb5-qrl-outputs` dataset.

> **Review fix (after the notebook-1 Kaggle run):** this cell first reused `nb1.rerank.alpha`. Notebook 1's cross-fit set that to 0 because plain beam search hurt MRR@25, and α = 0 would silently remove the Q-centroid from the ranking, so notebook 5 now has its own keys.

Instead of conditioning Stage 2 unimodal similarity on a single unconstrained beam sequence, average the `ChemFormerMolecularEncoder` embeddings of the top-3 mass-verified Q-beam structures:

$$\mathbf{E}^m_+ = \text{Normalize}\left(\sum_{j=1}^{3} w_j\, f_m(y_j^+; \gamma)\right), \qquad w_j = \frac{\exp(R_{\text{term}}(y_j^+))}{\sum_{k=1}^{3}\exp(R_{\text{term}}(y_k^+))}$$

This reward-weighted centroid sharpens $\cos(\mathbf{E}^m_+, \mathbf{E}^m_i)$ for known database isomers (Tier 2) while supplying chemically valid, mass-exact de novo candidates for novel structures (Tier 3).

In [ ]:
# [Colab Q-RL Cell 5] Stage 2.5 Q-learning on the competition data, Q-guided validation, test submission
class QRLCFG:
    RL_MINUTES = float(os.environ.get("RL77_RL_MINUTES", 45))
    N_TRAIN_MOLECULES = int(os.environ.get("RL77_NB5_TRAIN_MOLECULES", 20000))
    N_VAL_MOLECULES = int(os.environ.get("RL77_NB5_VAL_MOLECULES", 200))
    BATCH = 16
    LR = 3e-5
    CENTROID_TOP = 3
    GUARD_WINDOW = 100  # steps in the rollout-reward moving average used by the best-checkpoint guard


_candidate_formula = lru_cache(maxsize=None)(truth_formula)  # RDKit formula of a library SMILES (notebook 4 helper)


def trim_padding(batch: Dict, pad_id: int) -> Dict:
    """Drop all-PAD token columns (SMILES are padded to 256; the median is 39 characters)."""
    for key in ("mol_tokens_cls", "mol_tokens_dec"):
        if key in batch:
            used = int((batch[key] != pad_id).sum(dim=1).max())
            batch[key] = batch[key][:, :used]
    return batch


@torch.no_grad()
def qrl_predict(
    glmr_model, q_decoder, reward_engine, fg_head, mist_filter, tokenizer, loader,
    lib_smiles, lib_masses, lib_embs, collect: Optional[List[Dict]] = None,
    alpha: Optional[float] = None, insert_rank: Optional[int] = None,
) -> Dict[str, List[str]]:
    """Retrieval -> cross-fusion -> Q-guided beams; re-rank candidates with the reward-weighted
    centroid E^m_+ of the top Q-beam structures (PDF, final section); insert the best mass-consistent beam.
    alpha / insert_rank are notebook 5's own Evidently weights (`nb5.rerank.*`): notebook 1 fits its
    knobs for plain beam search, and a value of 0 there would silently switch the Q-centroid off here."""
    alpha = W("nb5.rerank.alpha") if alpha is None else alpha
    insert_rank = W.int("nb5.rerank.generated_insert_rank") if insert_rank is None else insert_rank
    rank_formulas = W.int("nb4.priors.rank_formulas")  # formula-prior ranking tiers adopted from notebook 4 (0 = off)
    lib_embs_dev = lib_embs.to(CFG.DEVICE)
    preds = {}
    for batch in loader:
        batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        h_s_mol, e_s_mol, _ = glmr_model.encode_molecule_spectra(batch)
        h_m_k, idx_batch = retrieve_top_k_candidates(
            e_s_mol, batch["precursor_mz"], batch["primary_adduct"], lib_masses, lib_embs,
            top_k=CFG.TOP_K_CANDIDATES, ionization_modes=batch["ionization_mode"],
        )
        h_ca = glmr_model.cross_fusion(h_s_mol, h_m_k)
        fg_71_pred, _ = fg_head(h_s_mol)
        priors = generate_formula_priors_batch(batch, mist_filter, fg_71_pred)
        neutral = np.array([
            neutral_mass_from_precursor(float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b])
            for b in range(len(batch["molecule_id"]))
        ], dtype=np.float32)
        rewards: List[List[float]] = []
        q_smis, q_iks = q_guided_beam_search_decode(
            q_decoder, reward_engine, h_ca, neutral, fg_71_pred.cpu().numpy(), priors, reward_lists=rewards,
        )
        for b, mol_id in enumerate(batch["molecule_id"]):
            cand_idx = idx_batch[b]
            cand_embs = h_m_k[b]
            stage2, gen_smi, gen_ik, gen_ok = None, None, None, False
            if q_smis[b]:
                top = q_smis[b][: QRLCFG.CENTROID_TOP]
                toks = torch.stack([tokenizer.encode(s, add_cls=True) for s in top]).to(CFG.DEVICE)
                _, e_top = glmr_model.mol_encoder(toks)
                w = torch.softmax(torch.tensor(rewards[b][: len(top)], device=CFG.DEVICE), dim=0)
                centroid = F.normalize((w.unsqueeze(1) * e_top).sum(dim=0), dim=-1)
                stage2 = torch.matmul(cand_embs, centroid).cpu().numpy()
                gen_smi, gen_ik = q_smis[b][0], q_iks[b][0]
                gen_ok = bool(in_mass_window(np.array([Descriptors.ExactMolWt(Chem.MolFromSmiles(gen_smi))]), float(neutral[b]))[0])
            fallback = torch.topk(torch.matmul(lib_embs_dev, e_s_mol[b]), k=min(100, len(lib_smiles))).indices.cpu().tolist()
            rec = {
                "molecule_id": str(mol_id),
                "cand_smiles": [lib_smiles[i] for i in cand_idx],
                "stage1_scores": torch.matmul(cand_embs, e_s_mol[b]).cpu().numpy(),
                "stage2_scores": stage2,
                "in_window": in_mass_window(lib_masses[cand_idx], float(neutral[b])),
                "gen_smiles": gen_smi, "gen_ik14": gen_ik, "gen_mass_ok": gen_ok,
                "backfill_smiles": [lib_smiles[i] for i in fallback],
                "q_beam_ik14s": list(q_iks[b]),
            }
            if rank_formulas > 0:
                rec["priority"] = formula_priority([_candidate_formula(s) for s in rec["cand_smiles"]], priors[b], rank_formulas)
            if collect is not None:
                collect.append(rec)
            preds[str(mol_id)] = assemble_ranked_candidates(rec, alpha, insert_rank)
    return preds


def run_qrl_pipeline() -> Dict:
    out_dir = ENV.work_dir()
    metrics: Dict = {"weights_version": W.version}
    nb1_dir = ENV.upstream_dir("nb1", "glmr_model.pt")
    lib_dir = ENV.upstream_dir("nb1", "library.parquet")
    split_dir = ENV.upstream_dir("nb1", "val_inchikey14.txt")
    nb3_dir = ENV.upstream_dir("nb3", "mass2smiles_fg_head.pt")

    glmr_model, tokenizer = load_glmr_checkpoint(nb1_dir / "glmr_model.pt", device=CFG.DEVICE)
    glmr_base, _ = load_glmr_checkpoint(nb1_dir / "glmr_model.pt", device=CFG.DEVICE)  # untouched baseline
    lib_smiles, lib_ik14, lib_masses, lib_embs = load_library(lib_dir)
    lib_embs = lib_embs.to(CFG.DEVICE)  # once: retrieve_top_k_candidates() otherwise copies 282 MB to the GPU per step
    fg_head = load_fg_head(nb3_dir)
    retriever = OfflineMassAndFormulaRetriever()
    mist_filter = FastMISTSubformulaFilter(CandidateFormulaBank(retriever.catalog_df["molecular_formula"].dropna().unique().tolist()))
    reward_engine = CASMI26ChemicalRewardEngine(tokenizer, canonicalize_and_inchikey14, cached_ecfp4_fg71)
    for p in glmr_model.parameters():
        p.requires_grad = False
    q_decoder = ChemFormerQDecoderWrapper(glmr_model.mol_decoder).to(CFG.DEVICE)  # fine-tunes glmr_model.mol_decoder in place
    for p in q_decoder.parameters():
        p.requires_grad = True
    trainer = GLMRStage25QLearningTrainer(q_decoder, reward_engine)
    optimizer = torch.optim.AdamW(
        [p for n, p in q_decoder.named_parameters() if not n.startswith("target_")], lr=QRLCFG.LR, weight_decay=1e-4
    )

    # 1. Stage 2.5 Q-learning on notebook 1's training split
    rng = np.random.default_rng(CFG.SEED)
    train_ids = read_split(split_dir, "train")
    train_ids = rng.choice(train_ids, size=min(QRLCFG.N_TRAIN_MOLECULES, len(train_ids)), replace=False).tolist()
    train_loader = make_loader(load_spectra_for_inchikeys(CFG.DATA_DIR, train_ids), tokenizer, train=True, batch_size=QRLCFG.BATCH)
    deadline = ENV.Deadline(QRLCFG.RL_MINUTES)
    log_rows, step = [], 0
    # Best-checkpoint guard: keep the decoder + Q-heads with the highest 100-step moving-average rollout reward
    # and restore it at the end, so late degradation cannot reach validation or the test submission.
    best = {"ma": -float("inf"), "step": None, "state": None}
    while not deadline.expired():
        for batch in train_loader:
            if deadline.expired():
                break
            stats = trainer.train_rl_step(
                trim_padding(batch, tokenizer.pad_id), glmr_model, fg_head, mist_filter, generate_formula_priors_batch,
                retrieve_top_k_candidates, lib_masses, lib_embs, ADDUCT_MASS_SHIFT, optimizer, CFG.DEVICE,
            )
            log_rows.append({"step": step, **stats})
            if step >= QRLCFG.GUARD_WINDOW - 1:
                ma = float(np.mean([r["mean_rollout_reward"] for r in log_rows[-QRLCFG.GUARD_WINDOW:]]))
                if ma > best["ma"]:
                    best = {"ma": ma, "step": step,
                            "state": {k: v.detach().to("cpu", copy=True) for k, v in q_decoder.state_dict().items()}}
            if step % 25 == 0:
                ENV.log(f"RL step {step}: {stats} ({deadline.remaining_min():.1f} min left)")
            step += 1
    pd.DataFrame(log_rows).to_csv(out_dir / "rl_log.csv", index=False)
    final_ma = float(np.mean([r["mean_rollout_reward"] for r in log_rows[-QRLCFG.GUARD_WINDOW:]])) if log_rows else None
    if best["state"] is not None and (final_ma is None or best["ma"] > final_ma):
        q_decoder.load_state_dict(best["state"])  # also restores glmr_model.mol_decoder (shared module)
        ENV.log(f"Restored the step-{best['step']} checkpoint (reward MA {best['ma']:.3f} vs final {final_ma:.3f})")
    torch.save({"state_dict": q_decoder.state_dict(), "weights_version": W.version, "rl_step": best["step"]}, out_dir / "q_decoder.pt")
    metrics["rl"] = {"steps": step, "mean_rollout_reward_last100": final_ma,
                     "best_reward_ma100": best["ma"] if best["state"] is not None else None, "best_step": best["step"],
                     "first_reward_ma100": float(np.mean([r["mean_rollout_reward"] for r in log_rows[:QRLCFG.GUARD_WINDOW]])) if log_rows else None}

    # 2. Validation: notebook 1's pipeline (its adopted re-rank knobs) vs Q-guided beams on the same held-out molecules
    val_ids = read_split(split_dir, "val")
    val_ids = rng.choice(val_ids, size=min(QRLCFG.N_VAL_MOLECULES, len(val_ids)), replace=False).tolist()
    val_df = load_spectra_for_inchikeys(CFG.DATA_DIR, val_ids, timstof_window_only=True)
    gt = ground_truth_ik14(val_df)
    base_sub = predict_and_rerank_test_molecules(
        glmr_base, make_loader(val_df, tokenizer, train=False, batch_size=16), tokenizer, lib_smiles, lib_masses, lib_embs,
    )
    records: List[Dict] = []
    q_preds = qrl_predict(glmr_model, q_decoder, reward_engine, fg_head, mist_filter, tokenizer,
                          make_loader(val_df, tokenizer, train=False, batch_size=8), lib_smiles, lib_masses, lib_embs, records)
    metrics["val"] = {
        "n": len(records),
        "mrr25_nb1_pipeline": compute_casmi_mrr25(predictions_from_submission(base_sub), gt),
        "mrr25_q_guided": compute_casmi_mrr25(q_preds, gt),
        "q_beam_any_valid": float(np.mean([r["gen_smiles"] is not None for r in records])),
        "q_beam_top1_mass_ok": float(np.mean([r["gen_mass_ok"] for r in records])),
        "q_beam_top1_exact": float(np.mean([r["gen_ik14"] == gt[r["molecule_id"]] for r in records])),
        "q_beam_contains_truth": float(np.mean([gt[r["molecule_id"]] in r["q_beam_ik14s"] for r in records])),
    }
    # Fit the Q-centroid blend and Q-beam insert rank (2-fold cross-fit, same grid as notebook 1)
    sweep = sweep_rerank_knobs(records, gt, prior_keys=("nb5.rerank.alpha", "nb5.rerank.generated_insert_rank"))
    metrics["rerank_sweep"] = {k: v for k, v in sweep.items() if k != "grid"}
    ENV.write_json(out_dir / "rerank_grid.json", sweep["grid"])
    ENV.write_json(out_dir / "fitted_weights.json", {
        "notebook": "nb5",
        "weights": {"nb5.rerank.alpha": sweep["best_alpha"],
                    "nb5.rerank.generated_insert_rank": sweep["best_generated_insert_rank"]},
        "evidence": {"metric": "MRR@25", "split": f"{len(records)} held-out timsTOF molecules, 2-fold cross-fit",
                     "prior": sweep["val_mrr25_prior"], "fitted_crossfit": sweep["val_mrr25_crossfit"]},
    })
    (alpha, insert_rank), used_fit = ENV.select_validated(
        (W("nb5.rerank.alpha"), W.int("nb5.rerank.generated_insert_rank")),
        (sweep["best_alpha"], sweep["best_generated_insert_rank"]),
        sweep["val_mrr25_prior"], sweep["val_mrr25_crossfit"],
    )
    metrics["test_rerank"] = {"alpha": alpha, "generated_insert_rank": insert_rank, "source": "fitted" if used_fit else W.version}
    ENV.log(f"Validation: {metrics['val']}; sweep {metrics['rerank_sweep']}; test uses {metrics['test_rerank']}")

    # 3. Test submission
    test_df = pd.read_parquet(CFG.DATA_DIR / "test.parquet")
    test_preds = qrl_predict(glmr_model, q_decoder, reward_engine, fg_head, mist_filter, tokenizer,
                             make_loader(test_df, tokenizer, train=False, batch_size=8), lib_smiles, lib_masses, lib_embs,
                             alpha=alpha, insert_rank=insert_rank)
    sub_df = pd.DataFrame([{"molecule_id": m, "smiles": ";".join(p)} for m, p in test_preds.items()], columns=["molecule_id", "smiles"])
    validate_submission_file(sub_df, test_df["molecule_id"].unique().tolist())
    sub_df.to_csv(out_dir / "submission.csv", index=False)

    ENV.write_json(out_dir / "metrics.json", metrics)
    ENV.write_run_manifest("nb5", out_dir, W.version, {"metrics": metrics})
    ENV.log(f"Notebook 5 outputs written to {out_dir}")
    return metrics


if __name__ == "__main__":
    NB5_METRICS = run_qrl_pipeline()

## How the Q-Learning Wrapper Elevates Stage 2 Re-Ranking & Tier 3 Novel Scaffolds

| Decoding Mechanism | Syntactic Validity | Exact Mass Match (within 15 ppm) | Functional Group Consistency | Impact on CASMI 2026 `MRR@25` |
| --- | --- | --- | --- | --- |
| **Standard Beam Search ($L_{gen}$ only)** | ~95% | 28% to 36% | Unconstrained | Generated SMILES $\mathbf{E}^m_+$ often drifts in molecular weight, adding noise to unimodal re-ranking. |
| **Q-Guided Beam Search ($L_{gen}$ + Dueling Q-RL)** | ~94% | **68% to 79%** | Enforced via 71-D FG reward | Produces multiple distinct `inchikey14` isomers matching exact precursor mass and neutral losses. |

*The table reproduces the PDF's expected ranges; `metrics.json` (`val.q_beam_*`, `val.mrr25_*`) holds what this run measured.*